# Task 3: Heart Disease Prediction
**DevelopersHub Corporation — AI/ML Engineering Internship**

## Problem Statement
Given a patient's health data (age, cholesterol, blood pressure, etc.), predict whether they are at risk of heart disease.
This is a **binary classification** problem: 0 = No Disease, 1 = Disease Present.

## Dataset
Heart Disease UCI Dataset — loaded directly via `ucimlrepo` (official UCI source).

## Goal
- Perform EDA to understand trends
- Train Logistic Regression and Decision Tree classifiers
- Evaluate using Accuracy, ROC-AUC, and Confusion Matrix
- Identify the most important features driving predictions

## Step 1: Install & Import Libraries

In [ ]:
# Install ucimlrepo if not already installed
!pip install ucimlrepo -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries imported successfully!')

## Step 2: Load the Dataset

In [ ]:
# Fetch Heart Disease dataset from UCI ML Repository (id=45)
heart_data = fetch_ucirepo(id=45)

# Combine features and target into one DataFrame
df = heart_data.data.features.copy()
df['target'] = heart_data.data.targets.values

# UCI dataset has target values 0-4; convert to binary (0 = no disease, 1 = disease)
df['target'] = df['target'].apply(lambda x: 1 if x > 0 else 0)

print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')
df.head()

## Step 3: Data Inspection

In [ ]:
# Column names and data types
print('=== Column Info ===')
print(df.info())

In [ ]:
# Summary statistics
print('=== Descriptive Statistics ===')
df.describe()

In [ ]:
# Check for missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')

In [ ]:
# Handle missing values if any (fill numeric with median)
df.fillna(df.median(numeric_only=True), inplace=True)
print('Missing values handled.')

In [ ]:
# Target class distribution
print('=== Target Distribution ===')
print(df['target'].value_counts())
print(f"\nDisease Rate: {df['target'].mean()*100:.1f}%")

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# --- Plot 1: Target Class Distribution ---
plt.figure(figsize=(7, 5))
ax = sns.countplot(x='target', data=df, palette='Set2')
ax.set_xticklabels(['No Disease (0)', 'Disease (1)'])
plt.title('Heart Disease Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Target')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 2: Age Distribution by Heart Disease Status ---
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='age', hue='target', bins=20, kde=True, palette='Set2')
plt.title('Age Distribution by Heart Disease Status', fontsize=14, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(title='Heart Disease', labels=['Disease', 'No Disease'])
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 3: Correlation Heatmap ---
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 4: Box Plots for Key Features by Target ---
key_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for i, feat in enumerate(key_features):
    sns.boxplot(x='target', y=feat, data=df, palette='Set2', ax=axes[i])
    axes[i].set_title(feat, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Heart Disease')
    axes[i].set_xticklabels(['No', 'Yes'])

plt.suptitle('Key Features vs Heart Disease (Box Plots)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot 5: Chest Pain Type vs Heart Disease ---
plt.figure(figsize=(8, 5))
sns.countplot(x='cp', hue='target', data=df, palette='Set2')
plt.title('Chest Pain Type vs Heart Disease', fontsize=14, fontweight='bold')
plt.xlabel('Chest Pain Type (0=Typical, 1=Atypical, 2=Non-anginal, 3=Asymptomatic)')
plt.ylabel('Count')
plt.legend(title='Heart Disease', labels=['No Disease', 'Disease'])
plt.tight_layout()
plt.show()

## Step 5: Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split (80/20, stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size: {X_train.shape[0]} samples')
print(f'Test set size: {X_test.shape[0]} samples')

## Step 6: Model Training

In [ ]:
# --- Model 1: Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

print('Logistic Regression trained!')
print(f'Accuracy: {accuracy_score(y_test, lr_preds)*100:.2f}%')

In [ ]:
# --- Model 2: Decision Tree ---
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)  # Decision Tree doesn't need scaling
dt_preds = dt_model.predict(X_test)
dt_proba = dt_model.predict_proba(X_test)[:, 1]

print('Decision Tree trained!')
print(f'Accuracy: {accuracy_score(y_test, dt_preds)*100:.2f}%')

## Step 7: Model Evaluation

In [ ]:
# --- Classification Reports ---
print('=' * 50)
print('LOGISTIC REGRESSION — Classification Report')
print('=' * 50)
print(classification_report(y_test, lr_preds, target_names=['No Disease', 'Disease']))

print('=' * 50)
print('DECISION TREE — Classification Report')
print('=' * 50)
print(classification_report(y_test, dt_preds, target_names=['No Disease', 'Disease']))

In [ ]:
# --- Confusion Matrices ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, lr_preds,
    display_labels=['No Disease', 'Disease'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Logistic Regression\nConfusion Matrix', fontsize=13, fontweight='bold')

ConfusionMatrixDisplay.from_predictions(
    y_test, dt_preds,
    display_labels=['No Disease', 'Disease'],
    cmap='Greens', ax=axes[1]
)
axes[1].set_title('Decision Tree\nConfusion Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- ROC Curves ---
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_proba)

lr_auc = roc_auc_score(y_test, lr_proba)
dt_auc = roc_auc_score(y_test, dt_proba)

plt.figure(figsize=(9, 6))
plt.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC = {lr_auc:.3f})', linewidth=2.5, color='steelblue')
plt.plot(dt_fpr, dt_tpr, label=f'Decision Tree (AUC = {dt_auc:.3f})', linewidth=2.5, color='seagreen')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC = 0.5)')
plt.fill_between(lr_fpr, lr_tpr, alpha=0.1, color='steelblue')
plt.fill_between(dt_fpr, dt_tpr, alpha=0.1, color='seagreen')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Logistic Regression ROC-AUC: {lr_auc:.4f}')
print(f'Decision Tree ROC-AUC:       {dt_auc:.4f}')

## Step 8: Feature Importance Analysis

In [ ]:
# --- Logistic Regression: Coefficient-based importance ---
lr_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': np.abs(lr_model.coef_[0])
}).sort_values('Coefficient', ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(lr_importance['Feature'], lr_importance['Coefficient'], color='steelblue')
plt.title('Logistic Regression — Feature Importance (|Coefficient|)', fontsize=13, fontweight='bold')
plt.xlabel('Absolute Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
# --- Decision Tree: Gini-based feature importance ---
dt_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(dt_importance['Feature'], dt_importance['Importance'], color='seagreen')
plt.title('Decision Tree — Feature Importance (Gini)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Step 9: Model Comparison Summary

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy (%)': [
        round(accuracy_score(y_test, lr_preds) * 100, 2),
        round(accuracy_score(y_test, dt_preds) * 100, 2)
    ],
    'ROC-AUC': [
        round(lr_auc, 4),
        round(dt_auc, 4)
    ]
})

print('=== Model Comparison ===')
print(summary.to_string(index=False))

## Step 10: Final Insights & Conclusions

### Key Findings:
1. **Best Performing Model**: Logistic Regression generally outperforms Decision Tree on this dataset due to the linear separability of health features.
2. **Most Important Features**:
   - `cp` (chest pain type) — strongest predictor of heart disease
   - `thalach` (maximum heart rate achieved) — higher rate correlates with disease
   - `oldpeak` (ST depression) — significant indicator
   - `ca` (number of major vessels) — more vessels colored = higher risk
3. **Age & Cholesterol**: Surprisingly, `age` alone is not the strongest predictor — heart rate and chest pain type matter more.
4. **Class Balance**: The dataset has a roughly balanced distribution between disease and no-disease cases, so accuracy is a reliable metric here.

### Conclusion:
The Logistic Regression model achieves strong predictive performance on heart disease classification. Chest pain type, maximum heart rate, and ST depression are the most clinically meaningful features in the model. This kind of ML-driven screening tool can assist doctors in early identification of at-risk patients.

> **Note**: This model is for educational purposes only and should not be used for actual medical diagnosis.